# Neural Network Simulator — interactive engineering notebook

**Aim:** assemble a *real trainable PyTorch network* from 1–64 specialized mathematical computational units. Compare outputs, gradients and accuracy. Here, "channels" is a user-facing alias for **algorithmic units**, not CNN feature-map channels.

**Flow:** numerical inputs `[batch, features]` → N heterogeneous algorithms → N module outputs → learned fusion → logits/probabilities.


In [ ]:
import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from neural_network_simulator import (
    ALGORITHMS, AlgorithmicNetwork, FunctionUnit, create_dataset, train_model
)
sns.set_theme(style="whitegrid")
print('PyTorch:', torch.__version__, '| Available algorithms:', ALGORITHMS)

## 1. Configure the architecture

Change the values and rerun cells. `CHANNELS` is an independent count of module outputs. `INPUTS` is a number of **numeric features**, not arbitrary raw media. All modules receive the same input vector in this version.

In [ ]:
CHANNELS = 12      # 1 to 64
INPUTS = 4         # numeric features per sample
SAMPLES = 600
EPOCHS = 120
SEED = 42
assert 1 <= CHANNELS <= 64
assert INPUTS >= 1
torch.manual_seed(SEED)
model = AlgorithmicNetwork(input_dim=INPUTS, channels=CHANNELS)
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))

## 2. Data and input/output contracts

We generate a nonlinear **toy binary classification** dataset. Every computational unit converts `[batch, INPUTS] → [batch]`. Fusion stacks those as `[batch, CHANNELS] → [batch]` logits.

In [ ]:
x, y = create_dataset(SAMPLES, INPUTS, seed=SEED)
split = int(0.8 * len(x))
x_train, y_train = x[:split], y[:split]
x_test, y_test = x[split:], y[split:]
print('Input:', tuple(x.shape), 'Labels:', tuple(y.shape))
print('Per-module outputs:', tuple(model.unit_outputs(x[:8]).shape))
print('Fused logits:', tuple(model(x[:8]).shape))

## 3. Train the neural network

All built-in units use differentiable PyTorch operators, so loss gradients propagate through both the fusion layer and the units. This is actual joint optimization (not simply a series of fixed Python functions).

In [ ]:
history = train_model(model, x_train, y_train, epochs=EPOCHS, learning_rate=0.03)
model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(x_test))
    predictions = (probabilities >= 0.5).float()
    test_accuracy = (predictions == y_test).float().mean().item()
print(f'Train loss: {history.losses[0]:.3f} -> {history.losses[-1]:.3f}')
print(f'Holdout accuracy: {test_accuracy:.1%}')

## 4. Inspect learning visually with Seaborn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(x=np.arange(1, len(history.losses)+1), y=history.losses, ax=axes[0])
axes[0].set(title='Training loss', xlabel='Epoch', ylabel='BCE with logits')
sns.lineplot(x=np.arange(1, len(history.accuracies)+1), y=history.accuracies, ax=axes[1])
axes[1].set(title='Training accuracy', xlabel='Epoch', ylabel='Accuracy', ylim=(0,1.05))
plt.tight_layout(); plt.show()

## 5. What is each computational unit actually returning?

The modules are diverse mathematical operations. They are **not** entire separately fitted sklearn algorithms by default; those can be integrated using adapters later.

In [ ]:
with torch.no_grad():
    module_matrix = model.unit_outputs(x_test[:12]).numpy()
labels = [f'{i+1}: {getattr(unit, "algorithm", "custom")}' for i, unit in enumerate(model.units)]
fig, ax = plt.subplots(figsize=(max(9, CHANNELS * 0.65), 5))
sns.heatmap(pd.DataFrame(module_matrix, columns=labels), cmap='vlag', center=0, ax=ax)
ax.set(xlabel='Algorithmic computational unit', ylabel='Input sample', title='Outputs from each unit')
plt.xticks(rotation=75, ha='right'); plt.tight_layout(); plt.show()

## 6. Create a custom mathematical module

The custom function below computes an input interaction. It remains differentiable because it uses torch operations. You can write other complex equations, use learnable parameters, or embed entire trainable torch sub-models this way.

In [ ]:
custom_units = [
    FunctionUnit(lambda batch: torch.sin(batch[:, 0] * batch[:, -1])),
    FunctionUnit(lambda batch: torch.tanh(batch.mean(dim=1))),
    FunctionUnit(lambda batch: (batch.square()).mean(dim=1)),
]
custom_model = AlgorithmicNetwork(INPUTS, channels=3, custom_units=custom_units)
custom_history = train_model(custom_model, x_train, y_train, epochs=80)
with torch.no_grad():
    custom_acc = ((custom_model(x_test) >= 0).float() == y_test).float().mean().item()
print('Custom module output shape:', tuple(custom_model.unit_outputs(x_test[:5]).shape))
print(f'Custom model holdout accuracy: {custom_acc:.1%}')

## 7. Can one unit contain another trainable neural network? Yes.

This subnetwork itself has multiple layers and learned parameters, but it emits **one scalar** for the outer fusion graph. Its weights will get trained through the full graph.

In [ ]:
torch.manual_seed(SEED)
nested = torch.nn.Sequential(torch.nn.Linear(INPUTS, 16), torch.nn.ReLU(), torch.nn.Linear(16, 1))
nested_model = AlgorithmicNetwork(INPUTS, 2, custom_units=[
    nested,
    FunctionUnit(lambda batch: torch.sin(batch[:, 0]))
])
nested_hist = train_model(nested_model, x_train, y_train, epochs=50)
print('Nested network gradient shape:', tuple(nested[0].weight.grad.shape))
print('Loss:', round(nested_hist.losses[0], 4), '->', round(nested_hist.losses[-1], 4))

## 8. A boundary: sklearn / CI/CD / hosted LLMs

A callable can supply predictions from sklearn or a hosted LLM, *after you encode them as numeric outputs*. But their inference or API calls are not differentiable PyTorch ops: gradients normally stop at that boundary. A CI/CD deployment should be executed by an external **orchestrator** after applying policies, not during gradient training.

Also, your GPU cannot automatically run an arbitrary Python function, Jenkins pipeline or hosted model. Device placement, kernels, serialization and network latency still matter.

**Next project versions:** plugin adapters for sklearn and remote inference, token/image encoders, trainable gating/routing, per-unit benchmarking, DAG topology, FastAPI model serving, MLflow and Kubernetes/Helm manifests.

## 9. Save the trained model state

Uncomment to save locally; to reload create the *same* architecture first, then `model.load_state_dict(torch.load(..., weights_only=True))`.

In [ ]:
# torch.save(model.state_dict(), 'simulator_state.pt')
print('Ready. Parameters:', sum(p.numel() for p in model.parameters()))